In [0]:
ruta_principal="/Volumes/olist/default/raw/"

In [0]:
%fs ls "/Volumes/olist/default/raw"

In [0]:
from pyspark.sql import functions as F

archivos_csv = [
    archivo
    for archivo in dbutils.fs.ls(ruta_principal)
    if archivo.name.lower().endswith(".csv")
]

print(f"Archivos encontrados: {len(archivos_csv)}")

In [0]:
dataframes = {}
total_por_archivo = {}
resumen = []

for archivo in archivos_csv:
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(archivo.path)
    )

    total_filas = df.count()

    dataframes[archivo.name] = df
    total_por_archivo[archivo.name] = total_filas

    resumen.append((
        archivo.name,
        archivo.size,
        total_filas,
        len(df.columns),
        ", ".join(df.columns)
    ))

df_resumen = spark.createDataFrame(
    resumen,
    ["archivo", "tamano_bytes", "filas", "columnas", "nombres_columnas"]
)

display(df_resumen)

In [0]:
for nombre_archivo, df in dataframes.items():
    print("=" * 80)
    print(f"ARCHIVO: {nombre_archivo}")
    print("=" * 80)

    df.printSchema()
    display(df.limit(5))

In [0]:
resumen_nulos = []

for nombre_archivo, df in dataframes.items():
    total_filas = total_por_archivo[nombre_archivo]

    expresiones = [
        F.sum(F.col(columna).isNull().cast("int")).alias(columna)
        for columna in df.columns
    ]

    resultado = df.agg(*expresiones).first().asDict()

    for columna, cantidad_nulos in resultado.items():
        porcentaje = (
            round((cantidad_nulos / total_filas) * 100, 2)
            if total_filas > 0 else 0
        )

        resumen_nulos.append((
            nombre_archivo,
            columna,
            cantidad_nulos,
            porcentaje
        ))

df_nulos = spark.createDataFrame(
    resumen_nulos,
    ["archivo", "columna", "cantidad_nulos", "porcentaje_nulos"]
)

display(df_nulos.orderBy(F.desc("porcentaje_nulos")))

In [0]:
resumen_ids = []

for nombre_archivo, df in dataframes.items():
    columnas_id = [
        columna
        for columna in df.columns
        if columna.lower().endswith("_id")
    ]

    for columna in columnas_id:
        resultado = (
            df.agg(
                F.count(F.col(columna)).alias("valores_no_nulos"),
                F.countDistinct(F.col(columna)).alias("valores_distintos")
            )
            .first()
        )

        resumen_ids.append((
            nombre_archivo,
            columna,
            resultado["valores_no_nulos"],
            resultado["valores_distintos"],
            resultado["valores_no_nulos"] - resultado["valores_distintos"]
        ))

df_ids = spark.createDataFrame(
    resumen_ids,
    [
        "archivo",
        "columna",
        "valores_no_nulos",
        "valores_distintos",
        "posibles_duplicados"
    ]
)

display(df_ids)